# Heart Disease Risk Prediction - Training Pipeline

## Problem Description
This notebook trains a K-Nearest Neighbors (KNN) model to predict the risk of heart disease (binary classification) based on 11 patient health metrics.

## Dataset
We use the **Heart Failure Prediction Dataset** (commonly found on Kaggle, originally compiled by fedesoriano). It contains 918 observations and 11 features.

## Methodology
1. **Preprocessing**: We apply one-hot encoding to categorical features (dropping the first category to avoid multicollinearity) and normalize the continuous features using `StandardScaler`.
2. **Model Selection**: KNN was chosen because it's a simple, interpretable distance-based algorithm that performs well on scaled tabular data of this size. We perform a hyperparameter grid search using 5-fold cross-validation to find the optimal number of neighbors ($k$).
3. **Evaluation**: The model is evaluated on a 20% stratified hold-out test set.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import joblib

### Load Dataset

In [2]:
# Load dataset
df = pd.read_csv('data/heart.csv')
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (918, 12)


### Preprocessing
We encode the categorical features and define the expected columns for the deployment pipeline.

In [3]:
# Split features and target
X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']

# One-hot encode categorical variables
X_encoded = pd.get_dummies(X, drop_first=True)

# Enforce exact feature order expected by Streamlit app
expected_columns = [
    'Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak', 
    'Sex_M', 'ChestPainType_ATA', 'ChestPainType_NAP', 'ChestPainType_TA', 
    'RestingECG_Normal', 'RestingECG_ST', 'ExerciseAngina_Y', 'ST_Slope_Flat', 'ST_Slope_Up'
]
X_encoded = X_encoded[expected_columns]

# Train/test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42, stratify=y)

# Scale numerical data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Model Training and Hyperparameter Tuning

In [4]:
# Hyperparameter tuning for KNN using GridSearchCV
knn = KNeighborsClassifier()
param_grid = {'n_neighbors': [3, 5, 7, 9, 11]}
grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train_scaled, y_train)

best_k = grid_search.best_params_['n_neighbors']
print(f"Best k found: {best_k}")

best_model = grid_search.best_estimator_

Best k found: 9


### Evaluation on Test Set

In [5]:
y_pred = best_model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.9021739130434783
Precision: 0.9038461538461539
Recall: 0.9215686274509803
F1: 0.9126213592233009
Confusion Matrix:
 [[72 10]
 [ 8 94]]
Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.88      0.89        82
           1       0.90      0.92      0.91       102

    accuracy                           0.90       184
   macro avg       0.90      0.90      0.90       184
weighted avg       0.90      0.90      0.90       184



### Save Model Artifacts
Export the trained model, scaler, and expected columns to be used by the Streamlit application.

In [6]:
joblib.dump(best_model, 'KNN_Heart.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(expected_columns, 'columns.pkl')
print("Artifacts saved successfully.")

Artifacts saved successfully.
